In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

In [ ]:
# ============================================================
# 1. EXTRACT - Carga de datos
# ============================================================
df_raw = pd.read_csv('penguins.csv')
print(f'Dimensiones originales: {df_raw.shape}')
print(f'Columnas: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
# ============================================================
# 2. EXPLORACION INICIAL
# ============================================================
print('--- Tipos de datos ---')
print(df_raw.dtypes)
print(f'\n--- Valores nulos por columna ---')
nulos = df_raw.isnull().sum()
print(nulos[nulos > 0])
print(f'\nTotal nulos: {nulos.sum()} de {df_raw.size} valores ({nulos.sum()/df_raw.size*100:.1f}%)')
print(f'\n--- Valores unicos en sex ---')
print(df_raw['sex'].value_counts(dropna=False))
print(f'\n--- Estadisticas descriptivas ---')
df_raw.describe()

In [ ]:
# ============================================================
# 3. TRANSFORM - Limpieza de datos
# ============================================================
df = df_raw.copy()

# Reemplazar 'NA' string y '.' por NaN real
df.replace(['NA', '.'], np.nan, inplace=True)

# Convertir columnas numericas que quedaron como object
cols_num = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']
for col in cols_num:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Eliminar filas con valores nulos
antes = len(df)
df = df.dropna()
print(f'Filas eliminadas por nulos: {antes - len(df)}')
print(f'Filas restantes: {len(df)}')

In [ ]:
# ============================================================
# 4. DETECCION Y TRATAMIENTO DE OUTLIERS (IQR)
# ============================================================
print('--- Deteccion de outliers por IQR ---')
mask_outlier = pd.Series(False, index=df.index)

for col in cols_num:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    outliers_col = (df[col] < lim_inf) | (df[col] > lim_sup)
    n_out = outliers_col.sum()
    if n_out > 0:
        print(f'{col}: {n_out} outliers fuera de [{lim_inf:.1f}, {lim_sup:.1f}]')
        print(f'  Valores: {df.loc[outliers_col, col].tolist()}')
    mask_outlier = mask_outlier | outliers_col

antes = len(df)
df = df[~mask_outlier]
print(f'\nFilas eliminadas por outliers: {antes - len(df)}')
print(f'Filas finales limpias: {len(df)}')

In [ ]:
# ============================================================
# 5. DISTRIBUCION DE VARIABLES ANTES DEL CLUSTERING
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
colores_hist = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for i, col in enumerate(cols_num):
    ax = axes[i // 2][i % 2]
    ax.hist(df[col], bins=25, color=colores_hist[i], edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('Frecuencia')
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1, label=f'Media: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='gray', linestyle=':', linewidth=1, label=f'Mediana: {df[col].median():.1f}')
    ax.legend(fontsize=8)

plt.suptitle('Distribucion de Variables Numericas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6. ESCALADO DE DATOS
# ============================================================
features = cols_num
x = df[features]

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

print('Estadisticas post-escalado (media ~ 0, std ~ 1):')
df_scaled = pd.DataFrame(x_scaled, columns=features)
print(df_scaled.describe().loc[['mean', 'std']].round(4))
print(f'\nDatos listos para clustering: {x_scaled.shape[0]} filas x {x_scaled.shape[1]} features')

In [ ]:
# ============================================================
# 7. METODO DEL CODO + SILHOUETTE
# ============================================================
K_rango = range(2, 10)
inercia = []
silhouettes = []

for k in K_rango:
    modelo = KMeans(n_clusters=k, n_init=10, random_state=42)
    etiquetas = modelo.fit_predict(x_scaled)
    inercia.append(modelo.inertia_)
    silhouettes.append(silhouette_score(x_scaled, etiquetas))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(K_rango, inercia, 'ro-', linewidth=2, markersize=8)
ax1.set_title('Metodo del Codo', fontsize=13, fontweight='bold')
ax1.set_xlabel('Numero de Clusters (K)')
ax1.set_ylabel('Inercia')
ax1.set_xticks(list(K_rango))
ax1.grid(True, alpha=0.3)
ax1.axvline(x=3, color='blue', linestyle='--', alpha=0.5, label='K=3 (codo)')
ax1.legend()

ax2.plot(K_rango, silhouettes, 'gs-', linewidth=2, markersize=8)
ax2.set_title('Coeficiente Silhouette', fontsize=13, fontweight='bold')
ax2.set_xlabel('Numero de Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(list(K_rango))
ax2.grid(True, alpha=0.3)
mejor_k = list(K_rango)[np.argmax(silhouettes)]
ax2.axvline(x=mejor_k, color='blue', linestyle='--', alpha=0.5, label=f'Mejor K={mejor_k} ({max(silhouettes):.3f})')
ax2.legend()

plt.tight_layout()
plt.show()

print('Silhouette por K:')
for k, s in zip(K_rango, silhouettes):
    marca = ' <-- optimo' if k == mejor_k else ''
    print(f'  K={k}: {s:.4f}{marca}')

In [ ]:
# ============================================================
# 8. MODELO FINAL: K-MEANS CON K=3
# ============================================================
K_FINAL = 3
kmeans = KMeans(n_clusters=K_FINAL, n_init=10, random_state=42)
df['cluster'] = kmeans.fit_predict(x_scaled)

print(f'Modelo entrenado con K={K_FINAL}')
print(f'Silhouette Score: {silhouette_score(x_scaled, df["cluster"]):.4f}')
print(f'Inercia final: {kmeans.inertia_:.2f}')
print(f'\nDistribucion de clusters:')
conteo = df['cluster'].value_counts().sort_index()
for c, n in conteo.items():
    print(f'  Cluster {c}: {n} pinguinos ({n/len(df)*100:.1f}%)')

In [ ]:
# ============================================================
# 9. PERFIL DE CADA CLUSTER
# ============================================================
print('--- Promedios por Cluster ---')
perfil = df.groupby('cluster')[cols_num].agg(['mean', 'std']).round(2)
print(perfil)

print('\n--- Composicion por Sexo ---')
sexo_cluster = pd.crosstab(df['cluster'], df['sex'], margins=True)
print(sexo_cluster)

In [ ]:
# ============================================================
# 10. VISUALIZACION PRINCIPAL: SCATTER PLOTS POR PARES
# ============================================================
colores_cluster = ['#e74c3c', '#2ecc71', '#3498db']
nombres = ['Cluster 0', 'Cluster 1', 'Cluster 2']

pares = [
    ('culmen_length_mm', 'flipper_length_mm', 'Longitud Pico vs Largo Aleta'),
    ('culmen_length_mm', 'culmen_depth_mm', 'Longitud Pico vs Profundidad Pico'),
    ('flipper_length_mm', 'body_mass_g', 'Largo Aleta vs Masa Corporal'),
    ('culmen_length_mm', 'body_mass_g', 'Longitud Pico vs Masa Corporal')
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (xvar, yvar, titulo) in enumerate(pares):
    ax = axes[idx // 2][idx % 2]
    for c in range(K_FINAL):
        grupo = df[df['cluster'] == c]
        ax.scatter(grupo[xvar], grupo[yvar],
                   c=colores_cluster[c], label=nombres[c],
                   alpha=0.7, edgecolors='k', linewidth=0.3, s=50)
    # Centroides en escala original
    centroides_orig = scaler.inverse_transform(kmeans.cluster_centers_)
    ix = cols_num.index(xvar)
    iy = cols_num.index(yvar)
    ax.scatter(centroides_orig[:, ix], centroides_orig[:, iy],
               c='black', marker='X', s=200, edgecolors='white', linewidth=2, label='Centroides', zorder=5)
    ax.set_xlabel(xvar)
    ax.set_ylabel(yvar)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('Agrupamiento Automatico de Pinguinos (K=3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 11. BOXPLOTS POR CLUSTER
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, col in enumerate(cols_num):
    datos = [df[df['cluster'] == c][col].values for c in range(K_FINAL)]
    bp = axes[i].boxplot(datos, patch_artist=True, labels=[f'C{c}' for c in range(K_FINAL)])
    for j, box in enumerate(bp['boxes']):
        box.set_facecolor(colores_cluster[j])
        box.set_alpha(0.7)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].grid(True, alpha=0.2)

plt.suptitle('Distribucion por Cluster', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 12. SILHOUETTE POR MUESTRA
# ============================================================
sil_vals = silhouette_samples(x_scaled, df['cluster'])
sil_avg = silhouette_score(x_scaled, df['cluster'])

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10

for c in range(K_FINAL):
    cluster_sil = np.sort(sil_vals[df['cluster'].values == c])
    y_upper = y_lower + len(cluster_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                     facecolor=colores_cluster[c], alpha=0.7, label=f'Cluster {c} (n={len(cluster_sil)})')
    ax.text(-0.05, y_lower + 0.5 * len(cluster_sil), str(c), fontweight='bold', fontsize=12)
    y_lower = y_upper + 10

ax.axvline(x=sil_avg, color='red', linestyle='--', linewidth=2, label=f'Promedio: {sil_avg:.3f}')
ax.set_title('Silhouette por Muestra', fontsize=13, fontweight='bold')
ax.set_xlabel('Coeficiente Silhouette')
ax.set_ylabel('Muestras')
ax.legend(loc='best')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 13. CENTROIDES EN ESCALA ORIGINAL
# ============================================================
centroides_orig = scaler.inverse_transform(kmeans.cluster_centers_)
df_centroides = pd.DataFrame(centroides_orig, columns=cols_num)
df_centroides.index.name = 'cluster'
print('--- Centroides (valores reales) ---')
print(df_centroides.round(2))

# Radar chart de centroides
categorias = cols_num
N = len(categorias)
angulos = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angulos += angulos[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for c in range(K_FINAL):
    # Normalizar cada centroide entre 0 y 1 para el radar
    valores = df_centroides.iloc[c].values
    valores_norm = (valores - df[cols_num].min().values) / (df[cols_num].max().values - df[cols_num].min().values)
    valores_norm = valores_norm.tolist()
    valores_norm += valores_norm[:1]
    ax.plot(angulos, valores_norm, 'o-', linewidth=2, label=f'Cluster {c}', color=colores_cluster[c])
    ax.fill(angulos, valores_norm, alpha=0.15, color=colores_cluster[c])

ax.set_xticks(angulos[:-1])
ax.set_xticklabels(['Largo\nPico', 'Prof.\nPico', 'Largo\nAleta', 'Masa\nCorporal'], fontsize=10)
ax.set_title('Perfil de Centroides (Radar)', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 14. RESUMEN FINAL
# ============================================================
print('='*60)
print('RESUMEN DEL AGRUPAMIENTO')
print('='*60)
print(f'Algoritmo: K-Means')
print(f'K seleccionado: {K_FINAL}')
print(f'Filas procesadas: {len(df)}')
print(f'Features: {cols_num}')
print(f'Escalado: StandardScaler')
print(f'Inercia: {kmeans.inertia_:.2f}')
print(f'Silhouette Score: {silhouette_score(x_scaled, df["cluster"]):.4f}')
print(f'\nInterpretacion de clusters:')
for c in range(K_FINAL):
    g = df[df['cluster'] == c]
    print(f'  Cluster {c}: {len(g)} pinguinos | '
          f'pico={g["culmen_length_mm"].mean():.1f}mm | '
          f'aleta={g["flipper_length_mm"].mean():.1f}mm | '
          f'masa={g["body_mass_g"].mean():.0f}g')